In [1]:
# Importamos Librerias
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
from sklearn.metrics import ( accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay)

In [6]:
## Cargamos el dataset Iris
from sklearn.datasets import load_iris
iris = load_iris(as_frame = True)

In [9]:
print(iris.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

In [10]:
dfIris = iris.frame

In [11]:
dfIris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


## Exploración del conjunto de datos

In [12]:
dfIris.shape

(150, 5)

In [15]:
dfIris.columns

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'target'],
      dtype='object')

In [17]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [18]:
## Renombrar la columna "target"

dfIris.rename(columns = { "target" : "especie" }, inplace = True)

In [19]:
dfIris["especie"] = dfIris["especie"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginiga"
})

In [21]:
dfIris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),especie
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [23]:
dfIris.tail()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),especie
145,6.7,3.0,5.2,2.3,virginiga
146,6.3,2.5,5.0,1.9,virginiga
147,6.5,3.0,5.2,2.0,virginiga
148,6.2,3.4,5.4,2.3,virginiga
149,5.9,3.0,5.1,1.8,virginiga


## Identificar variables predictoras y variables objetivo

In [24]:
# Variables predictoras 
X = dfIris.drop(columns = ["especie"])

In [28]:
print("Variables predictoras")
print(X.columns)

Variables predictoras
Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')


In [26]:
# Variable objetivo
Y = dfIris["especie"]

In [27]:
print("Variable objetivo")
print(Y.name)

Variable objetivo
especie


## Dividir los datos en prueba y entrenamiento

80% para entrenamiento

20% para prueba

In [32]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size = 0.20, random_state= 42, stratify= Y
)

In [34]:
print("X_train", X)

X_train      sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                  5.1               3.5                1.4               0.2
1                  4.9               3.0                1.4               0.2
2                  4.7               3.2                1.3               0.2
3                  4.6               3.1                1.5               0.2
4                  5.0               3.6                1.4               0.2
..                 ...               ...                ...               ...
145                6.7               3.0                5.2               2.3
146                6.3               2.5                5.0               1.9
147                6.5               3.0                5.2               2.0
148                6.2               3.4                5.4               2.3
149                5.9               3.0                5.1               1.8

[150 rows x 4 columns]


## Escalado de variables

In [36]:
scaler = StandardScaler()

In [37]:
# Escalamos las variables 

X_train_escalado = scaler.fit_transform(X_train)
X_test_escalado = scaler.transform(X_test)

In [38]:
print("Sin escalar")
print(X_train.head())
print("Escalado")
print(X_train_escalado[:5])

Sin escalar
     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
8                  4.4               2.9                1.4               0.2
106                4.9               2.5                4.5               1.7
76                 6.8               2.8                4.8               1.4
9                  4.9               3.1                1.5               0.1
89                 5.5               2.5                4.0               1.3
Escalado
[[-1.72156775 -0.33210111 -1.34572231 -1.32327558]
 [-1.12449223 -1.22765467  0.41450518  0.6517626 ]
 [ 1.14439475 -0.5559895   0.58484978  0.25675496]
 [-1.12449223  0.11567567 -1.28894078 -1.45494479]
 [-0.40800161 -1.22765467  0.13059752  0.12508575]]


## Entrenamiento del modelo KMN

En esta prueba usaremos k = 3

In [39]:
modelo = KNeighborsClassifier(n_neighbors= 3)

In [41]:
modelo.fit(
    X_train_escalado, Y_train
)

KNeighborsClassifier(n_neighbors=3)

## Realizamos predicciones con las variables de prueba

In [42]:
prediccion = modelo.predict(X_test_escalado)

In [45]:
print(prediccion[:10])

['setosa' 'virginiga' 'versicolor' 'versicolor' 'setosa' 'versicolor'
 'setosa' 'setosa' 'virginiga' 'versicolor']


In [46]:
## Comparacion entre valores reales y valores predecidos
comparacion = pd.DataFrame({
    "especie real": Y_test.values,
    "especie predicha": prediccion
})

In [47]:
comparacion.head(10)

,especie real,especie predicha
0,setosa,setosa
1,virginiga,virginiga
2,versicolor,versicolor
3,versicolor,versicolor
4,setosa,setosa
5,versicolor,versicolor
6,setosa,setosa
7,setosa,setosa
8,virginiga,virginiga
9,versicolor,versicolor


## Evaluacion del modelo

In [48]:
accuracy = accuracy_score(Y_test, prediccion)
print(f"accuracy/exactitud: {accuracy: .4f}")

accuracy/exactitud:  0.9333


In [49]:
## Matriz de confusion
cm = confusion_matrix(Y_test, prediccion)

In [50]:
cm_df = pd.DataFrame(
    cm,
    index = modelo.classes_,
    columns= modelo.classes_
)
cm_df

,setosa,versicolor,virginiga
setosa,10,0,0
versicolor,0,10,0
virginiga,0,2,8


## Clasification report
proporciona metricas que permiten evaluar el desempeño de cada clase: precision, recall, f1-score, support

In [51]:
print(classification_report( Y_test, prediccion))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginiga       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

